# Демо проектных упражнений модуля 6 — Обучаемые нормы и ось Q/K

**Демо-реализация, не готовое решение полномасштабного проекта.** Синтетическая регрессия: одинаковые данные, начальные линейные слои и оптимизатор; отдельная отложенная выборка. У RMSNorm есть шкала, у DyT/DyISRU — шкала, смещение и скаляр формы. Число параметров различается и выводится явно. Это учебный опыт, не сравнение языковых моделей.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])


## Демо 1. Как сравнить обучаемые нормы

Зададим одну задачу регрессии: по 32 признакам предсказываем sin(x₀)+0.2x₁. Первые 512 примеров используем для обучения, остальные 256 — для проверки. Линейные слои начинают с одинаковых весов. Меняется нормализатор, но вместе с ним меняется и число обучаемых параметров: код выводит его рядом с ошибкой.

Перед запуском разделите два вопроса: может ли модель уменьшить ошибку на обучении и переносится ли это улучшение на отложенные примеры? Поэтому рисуем две кривые, а не выбираем победителя по одной последней точке.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
torch.set_num_threads(1)
ru = DEMO_LANGUAGE == "ru"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#5E81AC", "#A3BE8C", "#B48EAD"])})
data_rng = torch.Generator().manual_seed(SEED)
x = torch.randn(768, 32, generator=data_rng) * torch.linspace(.5, 3., 32)
y = torch.sin(x[:, 0]) + .2*x[:, 1]
x_train, x_valid = x[:512], x[512:]
y_train, y_valid = y[:512], y[512:]

class RMSNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(32))
    def forward(self, z):
        return self.gain * z * torch.rsqrt(z.square().mean(-1, keepdim=True)+1e-6)

class DyT(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(.5))
        self.gain = nn.Parameter(torch.ones(32))
        self.bias = nn.Parameter(torch.zeros(32))
    def forward(self, z):
        return self.gain * torch.tanh(self.alpha*z) + self.bias

class DyISRU(nn.Module):
    def __init__(self):
        super().__init__()
        # b=4 makes the initial slope 1/sqrt(b)=0.5, matching DyT.
        self.raw_b = nn.Parameter(torch.log(torch.expm1(torch.tensor(4.))))
        self.gain = nn.Parameter(torch.ones(32))
        self.bias = nn.Parameter(torch.zeros(32))
    def forward(self, z):
        b = F.softplus(self.raw_b) + 1e-8
        return self.gain * z * torch.rsqrt(b+z.square()) + self.bias

def train(norm_cls):
    torch.manual_seed(SEED)  # These normalizers consume no random draws.
    model = nn.Sequential(nn.Linear(32,32), norm_cls(), nn.SiLU(), nn.Linear(32,1))
    initial = [p.detach().clone() for i in (0,3) for p in model[i].parameters()]
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    history = []
    for _ in range(80):
        model.train()
        loss = F.mse_loss(model(x_train).squeeze(-1), y_train)
        assert torch.isfinite(loss)
        opt.zero_grad(); loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        opt.step()
        model.eval()
        with torch.no_grad():
            train_mse = F.mse_loss(model(x_train).squeeze(-1), y_train)
            valid = F.mse_loss(model(x_valid).squeeze(-1), y_valid)
        history.append((train_mse.item(),valid.item()))
    return np.asarray(history),initial,sum(p.numel() for p in model.parameters())

fig,axes=plt.subplots(1,2,figsize=(11,4.8),dpi=150)
initial_reference = None
for cls in (RMSNorm,DyT,DyISRU):
    history,initial,count=train(cls)
    if initial_reference is None: initial_reference=initial
    for a,b in zip(initial,initial_reference): torch.testing.assert_close(a,b,atol=0,rtol=0)
    assert np.isfinite(history).all()
    for index,ax in enumerate(axes): ax.plot(np.arange(1,81),history[:,index],label=cls.__name__)
    print(cls.__name__, "parameters:",count,"final train/validation MSE:",history[-1])
for ax,title in zip(axes,["Обучение","Отложенная выборка"] if ru else ["Training","Held-out validation"]):
    ax.set_title(title,loc="left");ax.set_yscale("log")
    ax.set_xlabel("Шаг" if ru else "Step");ax.set_ylabel("MSE");ax.grid(alpha=.25);ax.legend()
fig.tight_layout();plt.show();plt.close(fig)

Сопоставьте разрыв между обучающей и проверочной ошибкой с числом параметров. Совпадение начальных линейных весов проверено явно, но единственная скорость обучения не обязана быть одинаково удачной для всех норм. Для более сильного вывода повторите опыт с несколькими скоростями и инициализациями; набор для окончательной оценки оставьте отдельно.

## Демо 2. Что меняет ось нормы Q/K

Сначала сравним скалярные произведения одних и тех же пар векторов: исходных, нормированных по RMS и единичных по L2. Это отдельные пары, а не вся матрица внимания. При ширине головы 64 RMS-вариант близок к восьмикратному косинусу; epsilon оставляет небольшую погрешность. Затем увеличим только вторую голову и предскажем, должна ли измениться первая при двух разных осях нормировки.

In [ ]:
# Paired scores isolate scale; they are not a complete attention matrix.
torch.manual_seed(SEED)
q = torch.randn(256, 8, 64) * 2.5
k = torch.randn(256, 8, 64) * 2.5
raw = (q*k).sum(-1)/math.sqrt(64)
q_n=F.rms_norm(q,[64],eps=1e-6);k_n=F.rms_norm(k,[64],eps=1e-6)
rms_scores=(q_n*k_n).sum(-1)/math.sqrt(64)
cosine=(F.normalize(q,dim=-1)*F.normalize(k,dim=-1)).sum(-1)
# A learned logit scale is intentionally omitted from this diagnostic.
torch.testing.assert_close(rms_scores,math.sqrt(64)*cosine,atol=2e-6,rtol=2e-6)
assert rms_scores.std() < raw.std()
print("raw / RMS / cosine std:",*[float(z.std()) for z in (raw,rms_scores,cosine)])
changed=q.clone();changed[:,1]*=10
per_head=F.rms_norm(changed,[64],eps=1e-6)
joint=F.rms_norm(q.flatten(-2),[8*64],eps=1e-6).reshape_as(q)
joint_changed=F.rms_norm(changed.flatten(-2),[8*64],eps=1e-6).reshape_as(q)
torch.testing.assert_close(per_head[:,0],q_n[:,0],atol=0,rtol=0)
assert not torch.allclose(joint[:,0],joint_changed[:,0])
print("Ось нормы меняет связь между головами." if ru else "The reduction axis changes coupling between heads.")

Независимая норма оставляет первую голову неизменной. Общая норма меняет её знаменатель, хотя сами её координаты не трогали. Различие вызвано осью редукции. Меньший разброс синтетических логитов сам по себе ещё не означает более качественное внимание или перевод.

## Переход к полному проекту

Для проекта с языковой моделью зафиксируйте токенизатор и порядок данных, проведите несколько запусков и перебор скорости обучения. Сравните качество, градиенты, время и память при одинаковом бюджете. Для реальных Q/K запишите ревизию модели, слой, ось нормы, обучаемые масштабы и стадию относительно RoPE; данный опыт использует только синтетические векторы.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.